In [ ]:
import os
import json
from pathlib import Path
from typing import List, Dict, Any
import networkx as nx
from tqdm import tqdm

# Import our custom entity extractor
from custom_entity_extractor import CustomEntityExtractor, TECHNICAL_PATTERNS

# Document loading and processing
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

class GraphBuilder:
    def __init__(self, entity_extractor: CustomEntityExtractor):
        self.entity_extractor = entity_extractor
        self.graph = nx.DiGraph()
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=200,
            length_function=len
        )
    
    def load_pdf(self, pdf_path: str) -> List[Dict[str, Any]]:
        """Load and split a PDF document into chunks."""
        loader = PyPDFLoader(pdf_path)
        pages = loader.load()
        
        chunks = []
        for page in pages:
            page_chunks = self.text_splitter.split_text(page.page_content)
            for i, chunk in enumerate(page_chunks):
                chunks.append({
                    'content': chunk,
                    'metadata': {
                        'source': pdf_path,
                        'page': page.metadata.get('page', 0),
                        'chunk_id': f"{os.path.basename(pdf_path)}_p{page.metadata.get('page', 0)}_c{i}"
                    }
                })
        return chunks
    
    def extract_chunk_entities(self, chunk: Dict[str, Any]) -> Dict[str, List[Dict[str, Any]]]:
        """Extract entities from a document chunk."""
        return self.entity_extractor.extract_entities(chunk['content'])
    
    def add_chunk_to_graph(self, chunk: Dict[str, Any], entities: Dict[str, List[Dict[str, Any]]]):
        """Add a document chunk and its entities to the graph."""
        chunk_id = chunk['metadata']['chunk_id']
        
        # Add document node
        self.graph.add_node(chunk_id, 
                           type='document',
                           content=chunk['content'],
                           metadata=chunk['metadata'])
        
        # Add entity nodes and CONTAINS relationships
        for entity_type, entity_list in entities.items():
            for entity in entity_list:
                entity_id = f"{entity_type}_{entity['text']}"
                
                # Add entity node if it doesn't exist
                if not self.graph.has_node(entity_id):
                    self.graph.add_node(entity_id,
                                      type='entity',
                                      entity_type=entity_type,
                                      text=entity['text'])
                
                # Add CONTAINS relationship
                self.graph.add_edge(chunk_id, entity_id,
                                  type='CONTAINS',
                                  start=entity['start'],
                                  end=entity['end'])
    
    def add_entity_relationships(self):
        """Add relationships between entities based on co-occurrence."""
        entity_nodes = [n for n, d in self.graph.nodes(data=True) if d['type'] == 'entity']
        
        for entity1 in entity_nodes:
            # Find documents containing entity1
            docs1 = set(n for n in self.graph.predecessors(entity1))
            
            for entity2 in entity_nodes:
                if entity1 >= entity2:  # Skip self-relationships and duplicates
                    continue
                    
                # Find documents containing entity2
                docs2 = set(n for n in self.graph.predecessors(entity2))
                
                # If entities co-occur in any documents, add RELATES_TO relationship
                common_docs = docs1 & docs2
                if common_docs:
                    self.graph.add_edge(entity1, entity2,
                                      type='RELATES_TO',
                                      weight=len(common_docs))
    
    def add_document_relationships(self):
        """Add relationships between document chunks."""
        doc_nodes = [n for n, d in self.graph.nodes(data=True) if d['type'] == 'document']
        
        for doc1 in doc_nodes:
            doc1_data = self.graph.nodes[doc1]
            
            for doc2 in doc_nodes:
                if doc1 >= doc2:  # Skip self-relationships and duplicates
                    continue
                    
                doc2_data = self.graph.nodes[doc2]
                
                # Add PART_OF relationship for chunks from same document
                if doc1_data['metadata']['source'] == doc2_data['metadata']['source']:
                    self.graph.add_edge(doc1, doc2,
                                      type='PART_OF',
                                      weight=1)
                
                # Add SIMILAR_TO relationship based on shared entities
                doc1_entities = set(n for n in self.graph.successors(doc1))
                doc2_entities = set(n for n in self.graph.successors(doc2))
                common_entities = doc1_entities & doc2_entities
                
                if common_entities:
                    self.graph.add_edge(doc1, doc2,
                                      type='SIMILAR_TO',
                                      weight=len(common_entities))
    
    def build_graph(self, pdf_dir: str):
        """Build the complete knowledge graph from a directory of PDFs."""
        pdf_files = list(Path(pdf_dir).glob('*.pdf'))
        
        # Process all PDFs
        for pdf_path in tqdm(pdf_files, desc="Processing PDFs"):
            # Load and split PDF
            chunks = self.load_pdf(str(pdf_path))
            
            # Process each chunk
            for chunk in tqdm(chunks, desc=f"Processing chunks from {pdf_path.name}", leave=False):
                # Extract entities
                entities = self.extract_chunk_entities(chunk)
                
                # Add to graph
                self.add_chunk_to_graph(chunk, entities)
        
        # Add relationships
        print("Adding entity relationships...")
        self.add_entity_relationships()
        
        print("Adding document relationships...")
        self.add_document_relationships()
        
        return self.graph

# Initialize the graph builder
builder = GraphBuilder(CustomEntityExtractor(custom_patterns=TECHNICAL_PATTERNS))

# Build the graph from PDFs in the data directory
graph = builder.build_graph('data/pdfs')

print("\nGraph Statistics:")
print(f"Number of nodes: {graph.number_of_nodes()}")
print(f"Number of edges: {graph.number_of_edges()}")

# Count node types
node_types = {}
for node, data in graph.nodes(data=True):
    node_type = data.get('type', 'unknown')
    node_types[node_type] = node_types.get(node_type, 0) + 1

print("\nNode types:")
for node_type, count in node_types.items():
    print(f"- {node_type}: {count}")

# Count edge types
edge_types = {}
for _, _, data in graph.edges(data=True):
    edge_type = data.get('type', 'unknown')
    edge_types[edge_type] = edge_types.get(edge_type, 0) + 1

print("\nEdge types:")
for edge_type, count in edge_types.items():
    print(f"- {edge_type}: {count}")
